In [70]:
import numpy as np

In [71]:
def H_matrix(t,Vt,M):   
    H_matrix = np.zeros((M, M), dtype=complex)
    for i in range(M):
        if i == 0:
            H_matrix[i,i], H_matrix[i,i+1] = Vt+4*t, -t
        elif i == M-1:
            H_matrix[i,i-1], H_matrix[i,i] = -t, Vt+4*t
        else:  
            H_matrix[i,i-1], H_matrix[i,i], H_matrix[i,i+1] = t, Vt+4*t, -t
    return H_matrix

In [72]:
def g_rs(t,M,N):   
    g_matrix = np.zeros((M, N), dtype=complex)
    for i in range(M):
        g_matrix[i,i] = t
    return g_matrix
g_rs(2,2,2)

array([[2.+0.j, 0.+0.j],
       [0.+0.j, 2.+0.j]])

In [73]:
def T_DL(t,M,N):   
    T_matrix = np.zeros((M, N), dtype=complex)
    for i in range(M):
        T_matrix[i,i] = t
    return T_matrix,T_matrix.conj().T


In [74]:
def G_kk(k,E,NJ,t,Vt):
    G_kk = np.zeros((k, k), dtype=complex)
    for i in range(k):
        if i==0:
            A=E*np.eye(k)+1j*NJ*np.eye(k)-H_matrix(t,Vt,k)[i,i]*np.eye(k)- T_DL(t,k,k)[0] @ g_rs(t,k,k) @ T_DL(t,k,k)[1]
            G_kk[i,i]=np.linalg.inv(A) 
        else:
            G_kk[i,i]=np.linalg.inv(E*np.eye(k)+1j*NJ*np.eye(k)-H_matrix(t,Vt,k)[i,i]*np.eye(k)- T_DL(t,k,k)[0] @ G_kk[i-1,i-1] @ T_DL(t,k,k)[1])
            G_kk[i-1,i]=G_kk[i-1,i-1] @ t*np.eye(k) @ G_kk[i,i]
    return G_kk

In [75]:
import numpy as np

def H_matrix(t, Vt, M):   
    H_matrix = np.zeros((M, M), dtype=complex)
    for i in range(M):
        if i == 0:
            H_matrix[i, i], H_matrix[i, i+1] = Vt + 4*t, -t
        elif i == M-1:
            H_matrix[i, i-1], H_matrix[i, i] = -t, Vt + 4*t
        else:  
            H_matrix[i, i-1], H_matrix[i, i], H_matrix[i, i+1] = t, Vt + 4*t, -t
    return H_matrix

def surface_green_dyson(E, eta, H00, H01, max_iter=100, tol=1e-8):
    """
    使用Dyson方程迭代法计算表面格林函数
    E: 能量
    eta: 小虚部（收敛因子）
    H00: 表面层哈密顿量
    H01: 层间耦合矩阵
    """
    # 初始猜测：表面格林函数 = 表面层孤立时的格林函数
    I = np.eye(H00.shape[0])
    G_surface = np.linalg.inv((E + 1j*eta) * I - H00)
    
    for i in range(max_iter):
        # Dyson方程: G = G0 + G0 * Σ * G
        # 其中自能 Σ = H01 * G_surface * H01ᴴ
        sigma = H01 @ G_surface @ H01.conj().T
        G_new = np.linalg.inv((E + 1j*eta) * I - H00 - sigma)
        
        # 检查收敛
        if np.max(np.abs(G_new - G_surface)) < tol:
            print(f"表面格林函数收敛于第 {i+1} 次迭代")
            break
            
        G_surface = G_new
    
    return G_surface

def T_DL(t, M, N):   
    T_matrix = np.zeros((M, N), dtype=complex)
    for i in range(min(M, N)):
        T_matrix[i, i] = t
    return T_matrix, T_matrix.conj().T

def G_kk_corrected(k, E, NJ, t, Vt):
    """
    正确的格林函数计算，使用迭代表面格林函数
    返回: k×k 的块矩阵，每个块是 k×k 矩阵
    """
    # 初始化：G_result[i,j] 每个都是 k×k 矩阵
    G_result = np.zeros((k, k, k, k), dtype=complex)
    
    # 预先计算所有矩阵
    H = H_matrix(t, Vt, k)  # k×k 矩阵
    T_mat, T_dagger = T_DL(t, k, k)  # 两个 k×k 矩阵
    
    # 计算表面格林函数（替代原来的 g_rs）
    H00 = H_matrix(t, Vt, k)  # 表面层哈密顿量
    H01 = t * np.eye(k)      # 层间耦合矩阵
    g_surface = surface_green_dyson(E, NJ, H00, H01)
    
    for i in range(k):
        # 获取对角元素（标量）
        diag_element = H[i, i]
        
        if i == 0:
            # 第一个格点：使用表面格林函数
            A = (E * np.eye(k) + 1j * NJ * np.eye(k) - 
                 diag_element * np.eye(k) - 
                 T_mat @ g_surface @ T_dagger)
            G_result[i, i] = np.linalg.inv(A)
            
        else:
            # 后续格点
            A = (E * np.eye(k) + 1j * NJ * np.eye(k) - 
                 diag_element * np.eye(k) - 
                 T_mat @ G_result[i-1, i-1] @ T_dagger)
            G_result[i, i] = np.linalg.inv(A)
            
            # 计算非对角元素
            G_result[i-1, i] = G_result[i-1, i-1] @ (t * np.eye(k)) @ G_result[i, i]
    
    return G_result

# 测试修正版本
try:
    print("\n=== 测试 k=3 ===")
    result3 = G_kk_corrected(3, 2, 10**(-5), 1, 1)
    print("k=3 计算成功！")
    print(f"结果形状: {result3.shape}")
    
except Exception as e:
    print(f"错误: {e}")
    import traceback
    traceback.print_exc()


=== 测试 k=3 ===
表面格林函数收敛于第 11 次迭代
k=3 计算成功！
结果形状: (3, 3, 3, 3)


In [78]:
k=5
E=2
NJ=10**(-5)
t=1
Vt=1
G=G_kk_corrected(k, E, NJ, t, Vt)
#print(G,G.shape)
H00=H_matrix(t, Vt, k)
H01=t * np.eye(k)
LL=T_DL(t, k, k)[0] @ surface_green_dyson(E, NJ, H00, H01, max_iter=100, tol=1e-8) @ T_DL(t, k, k)[1]
RR=T_DL(t, k, k)[0] @ surface_green_dyson(E, NJ, H00, H01, max_iter=100, tol=1e-8) @ T_DL(t, k, k)[1]
LL1=(LL-LL.conj())*1j
RR1=(RR-RR.conj())*1j

表面格林函数收敛于第 15 次迭代
表面格林函数收敛于第 15 次迭代
表面格林函数收敛于第 15 次迭代


In [79]:
new_11=LL1 @ G[0,k-1] @ RR1 @ G[0,k-1].conj().T
g=np.trace(new_11)
print(g)

0j
